In [82]:
import os
import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("hf_token")
PINECONE_API_KEY = os.getenv("pinecone_api_key")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("HF_TOKEN loaded:", HF_TOKEN is not None)
print("PINECONE_API_KEY loaded:", PINECONE_API_KEY is not None)
print("GROQ_API_KEY loaded:", GROQ_API_KEY is not None)

HF_TOKEN loaded: True
PINECONE_API_KEY loaded: True
GROQ_API_KEY loaded: True


In [ ]:
MODEL_PATH = "../iotid20/results/xgb_model.pkl"
LABEL_MAP_PATH = "../iotid20/results/label_mapping.json"   
DATA_PATH = load_dataset("KathiS/Final_Preprocessed_IoTID20", token=HF_TOKEN) 


Using the latest cached version of the dataset since KathiS/Final_Preprocessed_IoTID20 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Maruf\.cache\huggingface\datasets\KathiS___final_preprocessed_io_tid20\default\0.0.0\48c96054df34f3d762f533743ddfbccf1c314115 (last modified on Fri Nov 21 00:49:38 2025).


In [84]:
with open(MODEL_PATH, "rb") as f:
    xgb_model = pickle.load(f)

print("Model loaded successfully")
print(type(xgb_model))

Model loaded successfully
<class 'xgboost.sklearn.XGBClassifier'>


In [85]:
with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_to_id = json.load(f)

id_to_label = {v: k for k, v in label_to_id.items()}

print("Label to ID:")
print(label_to_id)

print("\nID to Label:")
print(id_to_label)

Label to ID:
{'DoS-Synflooding': 0, 'MITM ARP Spoofing': 1, 'Mirai-Ackflooding': 2, 'Mirai-HTTP Flooding': 3, 'Mirai-Hostbruteforceg': 4, 'Mirai-UDP Flooding': 5, 'Normal': 6, 'Scan Hostport': 7, 'Scan Port OS': 8}

ID to Label:
{0: 'DoS-Synflooding', 1: 'MITM ARP Spoofing', 2: 'Mirai-Ackflooding', 3: 'Mirai-HTTP Flooding', 4: 'Mirai-Hostbruteforceg', 5: 'Mirai-UDP Flooding', 6: 'Normal', 7: 'Scan Hostport', 8: 'Scan Port OS'}


In [86]:
df = DATA_PATH["train"].to_pandas()
print(df.shape)
print(df.columns.tolist()[:20])
df.head(2)

(625783, 26)
['Flow_Duration', 'Dst_Port', 'Flow_IAT_Min', 'Flow_IAT_Mean', 'Flow_IAT_Max', 'Flow_Pkts/s', 'Flow_Byts/s', 'Pkt_Len_Max', 'Pkt_Len_Mean', 'Pkt_Size_Avg', 'Pkt_Len_Min', 'Pkt_Len_Std', 'Idle_Min', 'Idle_Mean', 'Idle_Max', 'Fwd_Pkts/s', 'Bwd_Pkts/s', 'Init_Bwd_Win_Byts', 'ACK_Flag_Cnt', 'SYN_Flag_Cnt']


,Flow_Duration,Dst_Port,Flow_IAT_Min,Flow_IAT_Mean,Flow_IAT_Max,Flow_Pkts/s,Flow_Byts/s,Pkt_Len_Max,Pkt_Len_Mean,Pkt_Size_Avg,...,Bwd_Pkts/s,Init_Bwd_Win_Byts,ACK_Flag_Cnt,SYN_Flag_Cnt,Bwd_Header_Len,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,Bwd_Pkt_Len_Max,Bwd_Pkt_Len_Mean,Label
0,75,10101,75.0,75.0,75.0,26666.666667,32160000.0,1430.0,1280.666667,1921.0,...,13333.333333,-1,0,0,8,982.0,982.0,1430.0,1430.0,Mirai-Ackflooding
1,5310,554,1056.0,2655.0,4254.0,564.971751,0.0,0.0,0.000000,0.0,...,376.647834,14600,0,1,44,0.0,0.0,0.0,0.0,DoS-Synflooding


In [87]:
if hasattr(xgb_model, "feature_names_in_"):
    MODEL_FEATURES = list(xgb_model.feature_names_in_)
else:
    raise ValueError("The loaded model does not expose feature_names_in_. Please save/retrain the model with feature names.")

print("Number of model features:", len(MODEL_FEATURES))
print(MODEL_FEATURES)

Number of model features: 25
[np.str_('Flow_Duration'), np.str_('Dst_Port'), np.str_('Flow_IAT_Min'), np.str_('Flow_IAT_Mean'), np.str_('Flow_IAT_Max'), np.str_('Flow_Pkts/s'), np.str_('Flow_Byts/s'), np.str_('Pkt_Len_Max'), np.str_('Pkt_Len_Mean'), np.str_('Pkt_Size_Avg'), np.str_('Pkt_Len_Min'), np.str_('Pkt_Len_Std'), np.str_('Idle_Min'), np.str_('Idle_Mean'), np.str_('Idle_Max'), np.str_('Fwd_Pkts/s'), np.str_('Bwd_Pkts/s'), np.str_('Init_Bwd_Win_Byts'), np.str_('ACK_Flag_Cnt'), np.str_('SYN_Flag_Cnt'), np.str_('Bwd_Header_Len'), np.str_('Fwd_Pkt_Len_Max'), np.str_('Fwd_Pkt_Len_Min'), np.str_('Bwd_Pkt_Len_Max'), np.str_('Bwd_Pkt_Len_Mean')]


In [88]:
missing_features = [f for f in MODEL_FEATURES if f not in df.columns]

print("Missing features:", missing_features)

if missing_features:
    raise ValueError(f"Dataset is missing required model features: {missing_features}")

Missing features: []


In [89]:
sample_idx = random.randint(0, len(df) - 1)
row = df.iloc[sample_idx].copy()

print("Selected row index:", sample_idx)
row.head()

Selected row index: 529309


Flow_Duration      167
Dst_Port            80
Flow_IAT_Min     167.0
Flow_IAT_Mean    167.0
Flow_IAT_Max     167.0
Name: 529309, dtype: object

In [90]:
X_one = pd.DataFrame([row[MODEL_FEATURES].to_dict()], columns=MODEL_FEATURES)

print("Input shape:", X_one.shape)
X_one.head()

Input shape: (1, 25)


,Flow_Duration,Dst_Port,Flow_IAT_Min,Flow_IAT_Mean,Flow_IAT_Max,Flow_Pkts/s,Flow_Byts/s,Pkt_Len_Max,Pkt_Len_Mean,Pkt_Size_Avg,...,Fwd_Pkts/s,Bwd_Pkts/s,Init_Bwd_Win_Byts,ACK_Flag_Cnt,SYN_Flag_Cnt,Bwd_Header_Len,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,Bwd_Pkt_Len_Max,Bwd_Pkt_Len_Mean
0,167,80,167.0,167.0,167.0,11976.047904,119760.479042,20.0,6.666667,10.0,...,5988.023952,5988.023952,256,1,0,20,20.0,20.0,0.0,0.0


In [91]:
pred_id = int(xgb_model.predict(X_one)[0])

if hasattr(xgb_model, "predict_proba"):
    pred_proba = xgb_model.predict_proba(X_one)[0]
    confidence = float(np.max(pred_proba))
else:
    pred_proba = None
    confidence = None

pred_label = id_to_label.get(pred_id, f"UNKNOWN_{pred_id}")

print("Predicted ID:", pred_id)
print("Predicted Label:", pred_label)
print("Confidence:", confidence)

Predicted ID: 3
Predicted Label: Mirai-HTTP Flooding
Confidence: 0.9861308932304382


In [92]:
TRUE_LABEL_COLUMN_CANDIDATES = ["label", "Label", "attack", "attack_name", "y", "y_true"]

true_label_col = None
for c in TRUE_LABEL_COLUMN_CANDIDATES:
    if c in df.columns:
        true_label_col = c
        break

true_label = row[true_label_col] if true_label_col is not None else None

print("True label column:", true_label_col)
print("True label value:", true_label)

True label column: Label
True label value: Mirai-HTTP Flooding


In [93]:
def percentile_bin(series):

    q = series.quantile([0.2, 0.4, 0.6, 0.8]).values

    def label(v):
        if v == 0:
            return "zero"
        elif v <= q[0]:
            return "very_low"
        elif v <= q[1]:
            return "low"
        elif v <= q[2]:
            return "medium"
        elif v <= q[3]:
            return "high"
        else:
            return "very_high"

    return series.apply(label)


feature_levels = {}

for feat in MODEL_FEATURES:
    
    if feat in df.columns:
        feature_levels[feat] = percentile_bin(df[feat])


def describe_row(row_index):

    descriptions = []

    for feat in MODEL_FEATURES:

        value = df.loc[row_index, feat]

        level = feature_levels[feat].loc[row_index]

        name = feat.replace("_", " ").lower()

        sentence = f"{name} is {level} ({value})."

        descriptions.append(sentence)

    return descriptions


feature_sentences = describe_row(sample_idx)

for s in feature_sentences:
    print(s)

flow duration is high (167).
dst port is very_low (80).
flow iat min is very_high (167.0).
flow iat mean is high (167.0).
flow iat max is high (167.0).
flow pkts/s is low (11976.047904191617).
flow byts/s is low (119760.47904191616).
pkt len max is low (20.0).
pkt len mean is low (6.666666666666668).
pkt size avg is low (10.0).
pkt len min is zero (0.0).
pkt len std is very_high (11.547005383792516).
idle min is very_high (167.0).
idle mean is very_high (167.0).
idle max is high (167.0).
fwd pkts/s is medium (5988.023952095808).
bwd pkts/s is low (5988.023952095808).
init bwd win byts is medium (256).
ack flag cnt is medium (1).
syn flag cnt is zero (0).
bwd header len is low (20).
fwd pkt len max is medium (20.0).
fwd pkt len min is medium (20.0).
bwd pkt len max is zero (0.0).
bwd pkt len mean is zero (0.0).


In [94]:
def build_incident_packet_descriptive(
    row_index,
    predicted_label,
    confidence,
    true_label=None
):

    lines = []

    lines.append("Network intrusion alert detected.")
    lines.append("")
    lines.append(f"ML predicted attack type: {predicted_label}")
    lines.append(f"Prediction confidence: {confidence:.4f}")

    if true_label is not None:
        lines.append(f"Observed reference label: {true_label}")

    lines.append("")
    lines.append("Observed traffic behavior:")

    feature_sentences = describe_row(row_index)

    for s in feature_sentences:
        lines.append(f"- {s}")

    return "\n".join(lines)

In [95]:
incident_packet = build_incident_packet_descriptive(
    row_index=sample_idx,
    predicted_label=pred_label,
    confidence=confidence,
    true_label=true_label
)

print(incident_packet)

Network intrusion alert detected.

ML predicted attack type: Mirai-HTTP Flooding
Prediction confidence: 0.9861
Observed reference label: Mirai-HTTP Flooding

Observed traffic behavior:
- flow duration is high (167).
- dst port is very_low (80).
- flow iat min is very_high (167.0).
- flow iat mean is high (167.0).
- flow iat max is high (167.0).
- flow pkts/s is low (11976.047904191617).
- flow byts/s is low (119760.47904191616).
- pkt len max is low (20.0).
- pkt len mean is low (6.666666666666668).
- pkt size avg is low (10.0).
- pkt len min is zero (0.0).
- pkt len std is very_high (11.547005383792516).
- idle min is very_high (167.0).
- idle mean is very_high (167.0).
- idle max is high (167.0).
- fwd pkts/s is medium (5988.023952095808).
- bwd pkts/s is low (5988.023952095808).
- init bwd win byts is medium (256).
- ack flag cnt is medium (1).
- syn flag cnt is zero (0).
- bwd header len is low (20).
- fwd pkt len max is medium (20.0).
- fwd pkt len min is medium (20.0).
- bwd pkt 

In [96]:
incident_data = {
    "row_index": int(sample_idx),
    "predicted_id": pred_id,
    "predicted_label": pred_label,
    "confidence": confidence,
    "true_label": true_label,

    "incident_packet": incident_packet,  
    "incident_packet_type": "descriptive",

    "model_features": MODEL_FEATURES
}

incident_data

{'row_index': 529309,
 'predicted_id': 3,
 'predicted_label': 'Mirai-HTTP Flooding',
 'confidence': 0.9861308932304382,
 'true_label': 'Mirai-HTTP Flooding',
 'incident_packet': 'Network intrusion alert detected.\n\nML predicted attack type: Mirai-HTTP Flooding\nPrediction confidence: 0.9861\nObserved reference label: Mirai-HTTP Flooding\n\nObserved traffic behavior:\n- flow duration is high (167).\n- dst port is very_low (80).\n- flow iat min is very_high (167.0).\n- flow iat mean is high (167.0).\n- flow iat max is high (167.0).\n- flow pkts/s is low (11976.047904191617).\n- flow byts/s is low (119760.47904191616).\n- pkt len max is low (20.0).\n- pkt len mean is low (6.666666666666668).\n- pkt size avg is low (10.0).\n- pkt len min is zero (0.0).\n- pkt len std is very_high (11.547005383792516).\n- idle min is very_high (167.0).\n- idle mean is very_high (167.0).\n- idle max is high (167.0).\n- fwd pkts/s is medium (5988.023952095808).\n- bwd pkts/s is low (5988.023952095808).\n- in

### Retrieval stage

In [97]:
from huggingface_hub import InferenceClient
import numpy as np
from sentence_transformers import CrossEncoder

embed_client = InferenceClient(
    model="BAAI/bge-large-en-v1.5",
    token=HF_TOKEN
)

print("Embedding client ready")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")

Embedding client ready


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6147.95it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded


In [98]:
# Cell 18 — Embedding Function

def embed_text(text):

    emb = embed_client.feature_extraction(text)

    emb = np.array(emb)

    # L2 normalization
    emb = emb / np.linalg.norm(emb)

    return emb.tolist()

In [99]:
# Cell 19 — Embed the Incident Packet
incident_embedding = embed_text(incident_data["incident_packet"])

print("Embedding length:", len(incident_embedding))

Embedding length: 1024


### Pinecone detection retrieval

In [100]:
# Cell 22 — Connect to Pinecone

from pinecone import Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

DETECTION_INDEX_NAME = "cybersec-llm-rag"
MITIGATION_INDEX_NAME = "mitigation-vector-db"

detection_index = pc.Index(DETECTION_INDEX_NAME)
mitigation_index = pc.Index(MITIGATION_INDEX_NAME)

print("Connected to both Pinecone indexes")

Connected to both Pinecone indexes


In [101]:
# Cell 23 — Global vector search
def search_detection_global(index, query_vector, top_k=10):
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    return results["matches"]

In [102]:
# Cell 24 — Filtered vector search
def search_detection_filtered(index, query_vector, predicted_label, top_k=10):
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
        filter={
            "attack_true": {"$eq": predicted_label}
        }
    )
    return results["matches"]

In [103]:
# Cell 25 — Confidence-aware retrieval/ controller

def confidence_aware_retrieval(
    index,
    query_vector,
    predicted_label,
    confidence,
    top_k=10
):
    print(f"\n[Retrieval Mode] Confidence = {confidence:.4f}")

    if confidence >= 0.8:
        print("→ High confidence: FILTERED + GLOBAL")

        global_matches = search_detection_global(index, query_vector, top_k)
        filtered_matches = search_detection_filtered(index, query_vector, predicted_label, top_k)

    elif confidence >= 0.5:
        print("→ Medium confidence: GLOBAL prioritized")

        global_matches = search_detection_global(index, query_vector, top_k)
        filtered_matches = search_detection_filtered(index, query_vector, predicted_label, int(top_k/2))

    else:
        print("→ Low confidence: ONLY GLOBAL")

        global_matches = search_detection_global(index, query_vector, top_k)
        filtered_matches = []

    return global_matches, filtered_matches

In [104]:
# Cell 26 — Execution

global_matches, filtered_matches = confidence_aware_retrieval(
    detection_index,
    incident_embedding,
    predicted_label=incident_data["predicted_label"],
    confidence=incident_data["confidence"],
    top_k=10
)

print("Global matches:", len(global_matches))
print("Filtered matches:", len(filtered_matches))


[Retrieval Mode] Confidence = 0.9861
→ High confidence: FILTERED + GLOBAL
Global matches: 10
Filtered matches: 10


In [105]:
# Cell 27 — Quick inspection

if global_matches:
    print("Top global match:")
    print(global_matches[0]["id"])
    print(global_matches[0]["score"])
    print(global_matches[0]["metadata"])

Top global match:
IoTID20_306
0.704367638
{'attack_pred': 'MITM ARP Spoofing', 'attack_true': 'MITM ARP Spoofing', 'confidence': 0.295874, 'dataset': 'IoTID20', 'model': 'XGBoost', 'record_id': 'IoTID20_306', 'sample_type': 'low_confidence', 'summary': 'Network traffic analysis record.\n\nTraffic behavior summary:\nMedium connection duration with medium packet activity and zero traffic volume.\n\nConnection characteristics:\nFlow duration is medium.\nDestination port is very low.\n\nTraffic timing:\nPacket inter arrival time minimum is low.\nPacket inter arri', 'text_len': 791}


In [106]:
# Cell 28 — Merge and dedupe matches

def merge_and_dedupe_matches(global_matches, filtered_matches):
    merged = []
    seen_ids = set()

    # put filtered matches first so they get priority
    for match in filtered_matches + global_matches:
        match_id = match["id"]
        if match_id not in seen_ids:
            merged.append(match)
            seen_ids.add(match_id)

    return merged


merged_matches = merge_and_dedupe_matches(global_matches, filtered_matches)

print("Merged unique matches:", len(merged_matches))

Merged unique matches: 20


In [107]:
'''def balance_datasets(matches, max_per_dataset=2):

    selected = []
    dataset_counts = {}

    for m in matches:

        dataset = m["metadata"].get("dataset","unknown")

        if dataset_counts.get(dataset,0) < max_per_dataset:

            selected.append(m)

            dataset_counts[dataset] = dataset_counts.get(dataset,0) + 1

    return selected'''

'def balance_datasets(matches, max_per_dataset=2):\n\n    selected = []\n    dataset_counts = {}\n\n    for m in matches:\n\n        dataset = m["metadata"].get("dataset","unknown")\n\n        if dataset_counts.get(dataset,0) < max_per_dataset:\n\n            selected.append(m)\n\n            dataset_counts[dataset] = dataset_counts.get(dataset,0) + 1\n\n    return selected'

In [108]:
# Cell 29 — Dataset diversity limiter and Apply dataset diversity

def limit_per_dataset(matches, max_per_dataset=10):
    selected = []
    dataset_counts = {}

    for match in matches:
        dataset = match["metadata"].get("dataset", "UNKNOWN")

        if dataset_counts.get(dataset, 0) < max_per_dataset:
            selected.append(match)
            dataset_counts[dataset] = dataset_counts.get(dataset, 0) + 1

    return selected

diverse_matches = limit_per_dataset(merged_matches, max_per_dataset=3)

print("Diverse matches:", len(diverse_matches))
for m in diverse_matches[:5]:
    print(m["id"], m["metadata"].get("dataset"), m["metadata"].get("attack_true"))        
    


Diverse matches: 3
IoTID20_208 IoTID20 Mirai-HTTP Flooding
IoTID20_222 IoTID20 Mirai-HTTP Flooding
IoTID20_151 IoTID20 Mirai-HTTP Flooding


### Reranking

In [109]:
# Cell 33 — Prepare reranker inputs
 
def build_reranker_pairs(query_text, matches):
    pairs = []

    for match in matches:
        meta = match["metadata"]
        doc_text = meta.get("summary", "")

        if not doc_text:
            doc_text = str(meta)

        pairs.append((query_text, doc_text))

    return pairs

In [110]:
# Cell 34 — Rerank matches

def rerank_matches(query_text, matches, reranker_model, top_k=5):

    if not matches:
        return []

    pairs = build_reranker_pairs(query_text, matches)
    scores = reranker_model.predict(pairs)

    rescored = []

    for match, score in zip(matches, scores):

        new_match = {
            "id": match["id"],
            "score": match["score"],
            "metadata": match["metadata"],
            "rerank_score": float(score)
        }

        rescored.append(new_match)

    rescored = sorted(
        rescored,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return rescored[:top_k]

In [111]:
# Cell 35 — Run reranking
top_cases = rerank_matches(
    query_text=incident_data["incident_packet"],
    matches=diverse_matches,
    reranker_model=reranker,
    top_k=5
)

print("Top reranked cases:", len(top_cases))



#Cell 36 — Inspect final top cases
for i, case in enumerate(top_cases, start=1):
    meta = case["metadata"]
    print("=" * 80)
    print(f"Case {i}")
    print("ID:", case["id"])
    print("Vector score:", case.get("score"))
    print("Rerank score:", case.get("rerank_score"))
    print("Attack:", meta.get("attack_true"))
    print("Dataset:", meta.get("dataset"))
    print("Confidence:", meta.get("confidence"))
    print("Summary:", meta.get("summary"))

Top reranked cases: 3
Case 1
ID: IoTID20_151
Vector score: 0.697557449
Rerank score: -7.486239910125732
Attack: Mirai-HTTP Flooding
Dataset: IoTID20
Confidence: 0.3981043
Summary: Network traffic analysis record.

Traffic behavior summary:
High connection duration with medium packet activity and zero traffic volume.

Connection characteristics:
Flow duration is high.
Destination port is very low.

Traffic timing:
Packet inter arrival time minimum is low.
Packet inter arrival 
Case 2
ID: IoTID20_208
Vector score: 0.6982584
Rerank score: -7.515999794006348
Attack: Mirai-HTTP Flooding
Dataset: IoTID20
Confidence: 0.35431495
Summary: Network traffic analysis record.

Traffic behavior summary:
Very high connection duration with very low packet activity and medium traffic volume.

Connection characteristics:
Flow duration is very high.
Destination port is very high.

Traffic timing:
Packet inter arrival time minimum is medium.
Pac
Case 3
ID: IoTID20_222
Vector score: 0.698205
Rerank score: -

### Mitigation retrieval

In [112]:
# Cell 37 — Mitigation search by predicted attack
def search_mitigation_filtered(index, query_vector, predicted_label, top_k=8):
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
        filter={
            "attack_name": {"$eq": predicted_label}
        }
    )
    return results["matches"]

In [113]:
# Cell 38 — Fallback global mitigation search
def search_mitigation_global(index, query_vector, top_k=8):
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    return results["matches"]

In [114]:
# Cell 39 — Run mitigation retrieval
mitigation_query_text = f"""
Mitigation guidance for attack: {incident_data['predicted_label']}

Incident context:
{incident_data['incident_packet']}
""".strip()

mitigation_query_embedding = embed_text(mitigation_query_text)

mitigation_matches = search_mitigation_filtered(
    mitigation_index,
    mitigation_query_embedding,
    predicted_label=incident_data["predicted_label"],
    top_k=8
)

if len(mitigation_matches) == 0:
    mitigation_matches = search_mitigation_global(
        mitigation_index,
        mitigation_query_embedding,
        top_k=8
    )

print("Mitigation matches:", len(mitigation_matches))

Mitigation matches: 8


In [115]:
# Cell 40 — Rerank mitigation matches
top_mitigations = rerank_matches(
    query_text=mitigation_query_text,
    matches=mitigation_matches,
    reranker_model=reranker,
    top_k=5
)

print("Top mitigation items:", len(top_mitigations))



# Cell 41 — Inspect mitigation results
for i, item in enumerate(top_mitigations, start=1):
    meta = item["metadata"]
    print("=" * 80)
    print(f"Mitigation {i}")
    print("ID:", item["id"])
    print("Vector score:", item.get("score"))
    print("Rerank score:", item.get("rerank_score"))
    print("Attack name:", meta.get("attack_name"))
    print("Framework:", meta.get("framework"))
    print("Section:", meta.get("section"))
    print("Control/Technique ID:", meta.get("id"))
    print("Name:", meta.get("name"))
    print("Text:", meta.get("text"))

Top mitigation items: 5
Mitigation 1
ID: mitigation_104
Vector score: 0.634174347
Rerank score: 0.6370842456817627
Attack name: Mirai UDP Flooding
Framework: MITRE
Section: mitigation
Control/Technique ID: M1037
Name: Filter Network Traffic
Text: Mitigation Filter Network Traffic: Employ network appliances and endpoint software to filter ingress, egress, and lateral network traffic. This includes protocol-based filtering, enforcing firewall rules, and blocking or restricting traffic based on predefined conditions to limit adversary movement and data exfiltration. This mitigation can be implemented through the following measures: Ingress Traffic Filtering: Use Case - Configure network firewalls to allow traffic only from authorized IP add
Mitigation 2
ID: mitigation_33
Vector score: 0.634174347
Rerank score: -1.8551369905471802
Attack name: DoS Hulk
Framework: MITRE
Section: mitigation
Control/Technique ID: M1037
Name: Filter Network Traffic
Text: Mitigation Filter Network Traffic: Empl

In [116]:
def format_historical_cases(cases, predicted_label):
    lines = []
    lines.append("SIMILAR HISTORICAL CASES")

    for i, case in enumerate(cases, start=1):
        meta = case["metadata"]
        attack = meta.get("attack_true", "N/A")

        # 🔥 NEW: conflict detection
        if attack == predicted_label:
            relation = "CONSISTENT"
        else:
            relation = "CONFLICTING"

        lines.append(f"\nCase {i} ({relation}):")
        lines.append(f"Attack: {attack}")
        lines.append(f"Dataset: {meta.get('dataset')}")
        lines.append(f"Confidence: {meta.get('confidence')}")
        lines.append("Behavior:")
        lines.append(meta.get("summary", ""))

    return "\n".join(lines)

In [117]:
# Cell 43 — Build mitigation context

def format_mitigation_items(items):
    lines = []
    lines.append("MITIGATION KNOWLEDGE")

    for i, item in enumerate(items, start=1):
        meta = item["metadata"]

        lines.append(f"\nMitigation Item {i}:")
        lines.append(f"Attack Name: {meta.get('attack_name', 'N/A')}")
        lines.append(f"Framework: {meta.get('framework', 'N/A')}")
        lines.append(f"Section: {meta.get('section', 'N/A')}")
        lines.append(f"ID: {meta.get('id', 'N/A')}")
        lines.append(f"Name: {meta.get('name', 'N/A')}")
        lines.append("Content:")
        lines.append(meta.get("text", "No content available."))

    return "\n".join(lines)

In [118]:
# Cell 44 — Build complete retrieval context
historical_context = format_historical_cases(top_cases, predicted_label=incident_data["predicted_label"])
mitigation_context = format_mitigation_items(top_mitigations)

retrieved_context = "\n\n".join([
    historical_context,
    mitigation_context
])

print(retrieved_context[:5000])

SIMILAR HISTORICAL CASES

Case 1 (CONSISTENT):
Attack: Mirai-HTTP Flooding
Dataset: IoTID20
Confidence: 0.3981043
Behavior:
Network traffic analysis record.

Traffic behavior summary:
High connection duration with medium packet activity and zero traffic volume.

Connection characteristics:
Flow duration is high.
Destination port is very low.

Traffic timing:
Packet inter arrival time minimum is low.
Packet inter arrival 

Case 2 (CONSISTENT):
Attack: Mirai-HTTP Flooding
Dataset: IoTID20
Confidence: 0.35431495
Behavior:
Network traffic analysis record.

Traffic behavior summary:
Very high connection duration with very low packet activity and medium traffic volume.

Connection characteristics:
Flow duration is very high.
Destination port is very high.

Traffic timing:
Packet inter arrival time minimum is medium.
Pac

Case 3 (CONSISTENT):
Attack: Mirai-HTTP Flooding
Dataset: IoTID20
Confidence: 0.33890694
Behavior:
Network traffic analysis record.

Traffic behavior summary:
High connectio

### LLM stage

In [119]:
def build_final_prompt(incident_data, historical_context, mitigation_context):

    confidence = incident_data["confidence"]

    
    if confidence < 0.5:
        confidence_instruction = """
CONFIDENCE LEVEL: LOW

The ML prediction is uncertain and may be incorrect.

You MUST:
• Critically evaluate the predicted attack.
• Actively look for conflicting evidence in retrieved cases.
• Consider alternative attack types if supported by evidence.
"""
    elif confidence < 0.8:
        confidence_instruction = """
CONFIDENCE LEVEL: MODERATE

The ML prediction may be partially correct.

You SHOULD:
• Validate consistency with retrieved cases.
• Highlight any conflicting patterns.
• Consider alternative explanations if necessary.
"""
    else:
        confidence_instruction = """
CONFIDENCE LEVEL: HIGH

The ML prediction is likely reliable.

You SHOULD:
• Validate consistency with retrieved cases.
• Only challenge the prediction if strong contradictory evidence exists.
"""

    system_prompt = """
You are a senior cybersecurity SOC analyst specializing in network intrusion detection and incident analysis.

Your task is to analyze a network traffic incident using the following sources of evidence:

1. Machine learning prediction output
2. Retrieved historical network traffic cases
3. Retrieved mitigation knowledge mapped to security frameworks (MITRE ATT&CK and NIST)
4. Provide DETAILED mitigation recommendations using the provided mitigation knowledge.

IMPORTANT:
- Do NOT summarize into short bullet points only.
- Expand each mitigation with explanation.
- Use the provided mitigation descriptions to explain HOW and WHY the mitigation works.
- Reference mitigation IDs (e.g., M1031, RA-5) in your explanation.
- Explain the practical implementation of each mitigation in a real network environment.


STRICT ANALYSIS RULES

• Only use the information provided in the incident packet and retrieved evidence.
• Never invent network traffic features or behaviors that are not present.
• If evidence for a feature is missing, explicitly state: "No evidence available".
• Prioritize historical cases with higher similarity or rerank scores.
• Clearly distinguish between:
  - Confirmed indicators
  - Weak indicators
  - Missing or uncertain evidence.
• Treat the ML prediction as a hypothesis that must be validated using retrieved evidence.
• You are allowed to challenge or override the ML prediction ONLY if strong evidence supports an alternative attack type.
• The final response must be written as a professional SOC analyst report.

Your reasoning should focus on technical traffic behavior patterns and attack characteristics.
"""

    user_prompt = f"""
================ INCIDENT ALERT ================

Machine Learning Predicted Attack:
{incident_data['predicted_label']}

Prediction Confidence:
{confidence:.4f}

{confidence_instruction}

Observed Network Traffic Behavior:
{incident_data['incident_packet']}


================ RETRIEVED HISTORICAL CASES ================

These cases were retrieved using semantic similarity search and reranking.

{historical_context}


================ RETRIEVED MITIGATION KNOWLEDGE ================

{mitigation_context}


================ ANALYSIS TASK ================

Perform a structured security analysis of the incident using the retrieved evidence.

Step 1 — Explain the predicted attack type
Describe what the predicted attack typically represents in real network environments.

Step 2 — Evidence comparison
Compare the observed traffic behavior with patterns from the retrieved historical cases.

Step 3 — Feature-level analysis
Perform a structured comparison of traffic characteristics.

Step 4 — Indicators assessment
Identify:

• Strong indicators (clear matches with historical attack patterns)
• Weak indicators (partial matches)
• Missing or uncertain evidence

Step 5 — ML prediction validation and override analysis
• Evaluate whether the ML prediction is correct based on evidence.
• If conflicting evidence exists, clearly explain why.
• If a different attack type is more consistent, propose the alternative and justify it.

Step 6 — Defensive recommendations
Recommend mitigation actions based on the retrieved mitigation knowledge.

Step 7 — Risk evaluation
Estimate the severity of the incident.

Use this severity scale:

Low → reconnaissance or low-risk activity  
Medium → suspicious activity with moderate risk  
High → confirmed attack indicators  
Critical → active attack causing significant impact  


================ OUTPUT FORMAT ================

Produce the response strictly using this SOC report structure:

SOC INCIDENT REPORT

Predicted Attack:
Confidence Assessment:

Final Attack Assessment:
(Confirm or override the ML prediction with justification)

Feature Comparison Table:
Feature | Incident Behavior | Historical Pattern | Match Strength

Strong Indicators:
Weak or Uncertain Indicators:
Missing Evidence:

Relevant Historical Cases:
(Explain which cases are most similar and why)

Recommended Mitigations:
1. [Mitigation Name] (ID)
   - Explanation:
   - Why it works:
   - How to implement:

2. ...

Severity Assessment:

Analyst Notes:
(Provide concise expert reasoning)
"""

    return system_prompt, user_prompt

In [123]:
system_prompt, user_prompt = build_final_prompt(
    incident_data=incident_data,
    historical_context=historical_context,
    mitigation_context=mitigation_context
)

'''print("SYSTEM PROMPT:\n")
print(system_prompt)

print("\n" + "="*100 + "\n")

print("USER PROMPT:\n")
print(user_prompt[:7000])'''

'print("SYSTEM PROMPT:\n")\nprint(system_prompt)\n\nprint("\n" + "="*100 + "\n")\n\nprint("USER PROMPT:\n")\nprint(user_prompt[:7000])'

## LLM OUTPUT

In [ ]:
from groq import Groq
from dotenv import load_dotenv
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)

response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.4,
    max_tokens=1600
)

llm_output = response.choices[0].message.content

print("="*90)
print("SOC INCIDENT ANALYSIS REPORT")
print("="*90)
print(llm_output)

SOC INCIDENT ANALYSIS REPORT
SOC INCIDENT REPORT

Predicted Attack: Mirai-HTTP Flooding
Confidence Assessment: The machine learning model predicts the attack with a confidence level of 0.9861, which is considered high. This suggests that the model is likely reliable in its prediction.

Final Attack Assessment: 
Based on the observed network traffic behavior and the retrieved historical cases, the predicted attack type of Mirai-HTTP Flooding is confirmed. The traffic behavior, such as high flow duration, very low destination port, and high packet inter-arrival time, matches the patterns observed in the historical cases. Therefore, the ML prediction is validated and not overridden.

Feature Comparison Table:
| Feature | Incident Behavior | Historical Pattern | Match Strength |
| --- | --- | --- | --- |
| Flow Duration | High (167) | High/Very High | Strong |
| Destination Port | Very Low (80) | Very Low/Very High | Medium |
| Packet Inter-Arrival Time | High (167.0) | Low/Medium | Medium